<a href="https://colab.research.google.com/github/LorenzoBioinfo/import-bioinformatics/blob/main/notebooks/Extract_Backbone_sequence_ProteinMPNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Background

## La pipeline di protein design

La moderna progettazione computazionale di proteine segue tipicamente questa pipeline:

1. **RFDiffusion** → genera le coordinate del backbone (traccia dei Cα, senza sequenza)
2. **ProteinMPNN** → predice la sequenza ottimale per quel backbone
3. **AlphaFold2 / ESMFold** → valida il design predicendo nuovamente la struttura a partire dalla sequenza

I file PDB del backbone prodotti da RFDiffusion contengono solo informazioni geometriche ma nessuna sequenza.

Per poterli usare con ProteinMPNN o AlphaFold, è necessario estrarre la sequenza dalle coordinate ATOM leggendo i nomi dei residui a tre lettere e convertendoli nei corrispondenti codici a singola lettera.

---

# Residui canonici vs non canonici

Un amminoacido canonico è uno dei 20 residui standard codificati dal codice genetico (`ALA`, `GLY`, `SER`, ...). Nei file PDB questi compaiono nei record `ATOM`.

I residui non canonici sono versioni chimicamente modificate degli amminoacidi standard. Compaiono frequentemente nelle strutture sperimentali a causa di:

- Modificazioni post-traduzionali (fosforilazione, metilazione)
- Incorporazione di selenometionina durante la cristallografia a raggi X (`MSE` invece di `MET`)
- Crosslink chimici o amminoacidi non naturali in proteine ingegnerizzate

---

## Residui non canonici più comuni

| Residuo non canonico | AA parentale | Lettera | Modifica |
|---|---|---|---|
| `MSE` | `MET` | `M` | Selenometionina — molto comune nelle strutture X-ray |
| `SEP` | `SER` | `S` | Fosfoserina |
| `TPO` | `THR` | `T` | Fosfotreonina |
| `PTR` | `TYR` | `Y` | Fosfotirosina |
| `CSE` | `CYS` | `C` | Selenocisteina |
| `HYP` | `PRO` | `P` | Idrossiprolina |

---

# Problema pratico

Se questi residui vengono ignorati, `is_aa(standard=True)` li salta silenziosamente e si ottiene una sequenza più corta con gap.

Questo può causare crash o risultati errati nei tool downstream come ProteinMPNN.

La soluzione corretta è usare:

```python
is_aa(standard=False)

# 0 Setup


In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install biopython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.8 MB/s eta 0:00:00


## 1 Download file PDB generati da RFDiffusion

Scarichiamo i file PDB generati da RFDiffusion direttamente dalla repository GitHub ufficiale.
Salviamoli in una cartella.


In [2]:
import subprocess
import os

os.makedirs("rfdiffusion_examples", exist_ok=True)

subprocess.run([
    "git", "clone",
    "--filter=blob:none",
    "--sparse",
    "https://github.com/RosettaCommons/RFdiffusion.git",
    "rfdiffusion_examples"
], check=True)

subprocess.run([
    "git", "-C", "rfdiffusion_examples",
    "sparse-checkout", "set", "examples"
], check=True)


pdb_files = []
for root, dirs, files in os.walk("rfdiffusion_examples/examples"):
    for f in files:
        if f.endswith(".pdb"):
            pdb_files.append(os.path.join(root, f))

print(f"PDB trovati: {len(pdb_files)}")
for f in pdb_files:
    print(" ", f)

PDB trovati: 11
  rfdiffusion_examples/examples/input_pdbs/5TPN.pdb
  rfdiffusion_examples/examples/input_pdbs/tau_peptide.pdb
  rfdiffusion_examples/examples/input_pdbs/5an7.pdb
  rfdiffusion_examples/examples/input_pdbs/2KL8.pdb
  rfdiffusion_examples/examples/input_pdbs/insulin_target.pdb
  rfdiffusion_examples/examples/input_pdbs/nickel_symmetric_motif.pdb
  rfdiffusion_examples/examples/input_pdbs/peptide_complex_ideal_helix.pdb
  rfdiffusion_examples/examples/input_pdbs/7zkr_GABARAP.pdb
  rfdiffusion_examples/examples/input_pdbs/3IOL.pdb
  rfdiffusion_examples/examples/input_pdbs/1qys.pdb
  rfdiffusion_examples/examples/input_pdbs/1YCR.pdb


## 2 Leggiamo il primo PDB file
Proviamo prima con una prima struttura.

In [3]:
from Bio.PDB import PDBParser
parser = PDBParser(QUIET=True)


pdb_path = "rfdiffusion_examples/examples/input_pdbs/tau_peptide.pdb"
structure = parser.get_structure("design", pdb_path)
model = structure[0]

print(f"File: {pdb_path}")
print(f"Catene: {[c.id for c in model]}")



for chain in model:
    residues = list(chain.get_residues())
    aa_residues   = [r for r in residues if r.id[0] == " "]
    hetatm        = [r for r in residues if r.id[0] not in (" ", "W")]
    water         = [r for r in residues if r.id[0] == "W"]

    print(f"Catena {chain.id}:")
    print(f"  Residui standard : {len(aa_residues)}")
    print(f"  HETATM           : {len(hetatm)}")
    print(f"  Acque            : {len(water)}")

    # Atomi presenti per residuo, nei backbone RFDiffusion
    # tipicamente trovi solo N, CA, C, O (backbone) senza le catene laterali
    if aa_residues:
        sample = aa_residues[0]
        atoms = [a.name for a in sample.get_atoms()]
        print(f"  Atomi nel primo residuo: {atoms}")


    print(f"  Primi 10 residui: {[r.resname for r in aa_residues[:10]]}")

File: rfdiffusion_examples/examples/input_pdbs/tau_peptide.pdb
Catene: ['A', 'B']
Catena A:
  Residui standard : 164
  HETATM           : 0
  Acque            : 0
  Atomi nel primo residuo: ['N', 'CA', 'C', 'O']
  Primi 10 residui: ['GLY', 'GLY', 'GLY', 'GLY', 'GLY', 'GLY', 'GLY', 'GLY', 'GLY', 'GLY']
Catena B:
  Residui standard : 14
  HETATM           : 0
  Acque            : 0
  Atomi nel primo residuo: ['N', 'CA', 'C', 'O']
  Primi 10 residui: ['GLY', 'GLY', 'GLY', 'LYS', 'VAL', 'GLN', 'ILE', 'ILE', 'ASN', 'LYS']


La cosa più importante da osservare è la riga "Atomi nel primo residuo". In un PDB sperimentale normale troveresti tutti gli atomi delle catene laterali (CB, CG, CD...). In un backbone RFDiffusion dovresti trovare solo ['N', 'CA', 'C', 'O'] perché RFDiffusion genera solo il backbone, senza catene laterali. Questo è esattamente il motivo per cui serve ProteinMPNN, per predire le sequenze su questi backbone.



Il file tau_peptide.pdb è un tipico output di binder design con RFDiffusion. La struttura ha due catene con ruoli completamente diversi:
Catena A — il binder generato (164 residui)
Tutti GLY placeholder — RFDiffusion ha generato la geometria backbone (N, CA, C, O) senza assegnare una sequenza reale. Questa è la catena che passeremo a ProteinMPNN per la sequenza design.
Catena B — il target fisso (14 residui)
Il peptide tau originale, entrato nel modello con la sua sequenza reale (LYS, VAL, GLN...). RFDiffusion lo ha tenuto fermo come riferimento geometrico per generare il binder.
Entrambe le catene hanno solo ['N', 'CA', 'C', 'O'] — nessuna catena laterale, nessun HETATM, nessuna acqua. È un backbone pulito, pronto per la pipeline downstream

## 3 Otteniamo la sequenza

Cerchiamo ora di ottenere la sequenza aminoacidica dal file PDB.
Il file potrebbe contenere anche AA non canonici, mappiamoli con quelli canonici.

In [4]:
from Bio.PDB.Polypeptide import protein_letters_3to1


NON_CANONICAL = {
    "MSE": "M",   # selenomethionine → methionine
    "SEP": "S",   # phosphoserine → serine
    "TPO": "T",   # phosphothreonine → threonine
    "PTR": "Y",   # phosphotyrosine → tyrosine
    "CSE": "C",   # selenocysteine → cysteine
    "HYP": "P",   # hydroxyproline → proline
    "MLY": "K",   # methylated lysine → lysine
    "FME": "M",   # formyl-methionine → methionine
}

RESIDUE_MAP = {**protein_letters_3to1, **NON_CANONICAL}

def resname_to_single(resname):
    """Convert three-letter residue name to single-letter. Unknown → X."""
    return RESIDUE_MAP.get(resname.upper(), "X")


for three, one in NON_CANONICAL.items():
    print(f"  {three} → {resname_to_single(three)}")

print(f"\n  ALA → {resname_to_single('ALA')}")
print(f"  GLY → {resname_to_single('GLY')}")
print(f"  UNK → {resname_to_single('UNK')}")

  MSE → M
  SEP → S
  TPO → T
  PTR → Y
  CSE → C
  HYP → P
  MLY → K
  FME → M

  ALA → A
  GLY → G
  UNK → X


In [6]:
from Bio.PDB.Polypeptide import is_aa

def extract_sequence(pdb_file, chain_id="A"):
    """
    Extract amino acid sequence from ATOM coordinates of a PDB file.

    - Includes non-canonical residues via is_aa(standard=False)
    - Maps non-canonical to single-letter via RESIDUE_MAP
    - Returns dict with sequence, non-canonical residues found, and a
      is_placeholder flag (True if chain is all GL, ex RFDiffusion backbone)
    """
    structure = parser.get_structure("s", pdb_file)
    model = structure[0]

    if chain_id not in [c.id for c in model]:
        raise ValueError(f"Chain {chain_id} not found in {pdb_file}")

    sequence    = []
    non_canon   = {}

    for residue in model[chain_id]:
        if not is_aa(residue, standard=False):  ## not IS_aa ci assicura che lavoriamo solo con le proteine
            continue

        resname = residue.resname.strip()
        single  = resname_to_single(resname)
        sequence.append(single)


        if resname in NON_CANONICAL:
            non_canon[resname] = non_canon.get(resname, 0) + 1

    seq_str = "".join(sequence)


    gly_frac = seq_str.count("G") / len(seq_str) if seq_str else 0
    is_placeholder = gly_frac > 0.9

    return {
        "sequence"       : seq_str,
        "length"         : len(seq_str),
        "non_canonical"  : non_canon,
        "is_placeholder" : is_placeholder,
        "gly_fraction"   : round(gly_frac, 2),
    }



for chain in ["A", "B"]:
    result = extract_sequence("rfdiffusion_examples/examples/input_pdbs/tau_peptide.pdb", chain_id=chain)
    print(f"Catena {chain}:")
    print(f"  Sequenza        : {result['sequence'][:30]}...")
    print(f"  Lunghezza       : {result['length']}")
    print(f"  Non canonici    : {result['non_canonical']}")
    print(f"  Is placeholder  : {result['is_placeholder']}")
    print(f"  GLY fraction    : {result['gly_fraction']}")
    print()

Catena A:
  Sequenza        : GGGGGGGGGGGGGGGGGGGGGGGGGGGGGG...
  Lunghezza       : 164
  Non canonici    : {}
  Is placeholder  : True
  GLY fraction    : 1.0

Catena B:
  Sequenza        : GGGKVQIINKKLDL...
  Lunghezza       : 14
  Non canonici    : {}
  Is placeholder  : False
  GLY fraction    : 0.21



## Eseguiamo il loop su tutti i file PDB

Per ogni PDB file, estraiamo la sequenza. Per comodità non conserviamo i placeholder di sola GLY.

In [7]:
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio import SeqIO
import os

all_records = []
report_rows = []

for pdb_path in pdb_files:
    pdb_name = os.path.basename(pdb_path).replace(".pdb", "")
    structure = parser.get_structure("s", pdb_path)
    model = structure[0]

    for chain in model:
        chain_id = chain.id

        try:
            result = extract_sequence(pdb_path, chain_id=chain_id)
        except ValueError as e:
            print(f" {e}")
            continue


        record_id = f"{pdb_name}_chain{chain_id}"


        report_rows.append({
            "pdb"            : pdb_name,
            "chain"          : chain_id,
            "length"         : result["length"],
            "is_placeholder" : result["is_placeholder"],
            "gly_fraction"   : result["gly_fraction"],
            "non_canonical"  : str(result["non_canonical"]),
        })


        if result["is_placeholder"]:
            print(f" Skipping {record_id}: placeholder backbone")
            continue

        rec = SeqRecord(
            Seq(result["sequence"]),
            id=record_id,
            description=f"extracted from {pdb_path}"
        )
        all_records.append(rec)


fasta_out = "designs_sequences.fasta"
SeqIO.write(all_records, fasta_out, "fasta")

print(f"\nSequenze salvate nel FASTA : {len(all_records)}")
print(f"File                       : {fasta_out}")

 Skipping tau_peptide_chainA: placeholder backbone

Sequenze salvate nel FASTA : 17
File                       : designs_sequences.fasta


## Report Finale

In [8]:

import pandas as pd

report_df = pd.DataFrame(report_rows)

real     = report_df[report_df["is_placeholder"] == False]
placeholders = report_df[report_df["is_placeholder"] == True]

print("=" * 45)
print("  SEQUENCE EXTRACTION REPORT")
print("=" * 45)
print(f"  PDB processati         : {report_df['pdb'].nunique()}")
print(f"  Catene totali          : {len(report_df)}")
print(f"  Sequenze reali         : {len(real)}")
print(f"  Backbone placeholder   : {len(placeholders)}")
print()
print(f"  Lunghezza media        : {real['length'].mean():.1f} aa")
print(f"  Lunghezza minima       : {real['length'].min()} aa")
print(f"  Lunghezza massima      : {real['length'].max()} aa")
print()


import ast
all_non_canon = {}
for val in report_df["non_canonical"]:
    d = ast.literal_eval(val)
    for k, v in d.items():
        all_non_canon[k] = all_non_canon.get(k, 0) + v

if all_non_canon:
    print("  Residui non canonici trovati:")
    for resname, count in sorted(all_non_canon.items(), key=lambda x: -x[1]):
        single = resname_to_single(resname)
        print(f"    {resname} → {single} : {count} occorrenze")
else:
    print("  Residui non canonici: nessuno trovato")

print()
print("=" * 45)


report_df.to_csv("extraction_report.csv", index=False)
print(f"  Report salvato: extraction_report.csv")
print("=" * 45)

  SEQUENCE EXTRACTION REPORT
  PDB processati         : 11
  Catene totali          : 18
  Sequenze reali         : 17
  Backbone placeholder   : 1

  Lunghezza media        : 122.1 aa
  Lunghezza minima       : 12 aa
  Lunghezza massima      : 490 aa

  Residui non canonici trovati:
    MSE → M : 1 occorrenze

  Report salvato: extraction_report.csv


## Esercizi


###Esercizio 1 - File PDB con catene mancanti

Alcuni file PDB potrebbero non avere le chain che ci aspettiamo. Modifica la funzione in modo che non crashi ma che restituisca un errore


###Esercizio 2 - Estendi il non-canonical mapping
Abbiamo utilizzato gli aa non canonici più comuni. Modifica il dizionario per aggiungerne altri da wwPDB Chemical Component Dictionary

###Esercizio 3 - Filtra per lunghezza
Aggiungi dei filtri prima di salvare la sequenza. Ad esempio, imponi una lunghezza minima o massima.